# Food Plate/Bowl Detector
**Multi-class Image Classification using Transfer Learning (EfficientNetB0)**

**Classes:**
- `food_on_plate` — food served on a plate
- `food_in_bowl` — food served in a bowl
- `not_plate_not_bowl` — food not on plate or bowl
- `empty_plate` — empty/clean plate
- `empty_bowl` — empty/clean bowl
- `hard_negatives` — ambiguous / mixed cases

**Pipeline:**
1. Dataset download from DuckDuckGo / Bing (via `icrawler`)
2. Data augmentation & preprocessing
3. EfficientNetB0 transfer learning with fine-tuning
4. Accuracy & Loss plots
5. Evaluation on test set
6. Model & history save

## 1. Install Dependencies

In [ ]:
!pip install -q icrawler tensorflow matplotlib scikit-learn seaborn

## 2. Imports

In [ ]:
import os
import json
import shutil
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 6. Sample Images Preview

In [ ]:
fig, axes = plt.subplots(len(CLASS_NAMES), 4, figsize=(14, 3 * len(CLASS_NAMES)))
fig.suptitle("Sample Images per Class", fontsize=16, fontweight="bold")

for row_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = RAW_DIR / class_name
    images = list(class_dir.glob("*.jpg")) + list(class_dir.glob("*.png")) + list(class_dir.glob("*.jpeg"))
    sample = random.sample(images, min(4, len(images)))
    for col_idx in range(4):
        ax = axes[row_idx][col_idx]
        if col_idx < len(sample):
            try:
                img = mpimg.imread(str(sample[col_idx]))
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, "Error", ha="center")
        else:
            ax.axis("off")
        if col_idx == 0:
            ax.set_ylabel(class_name, fontsize=9, rotation=45, ha="right")
        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.savefig("sample_images.png", dpi=120)
plt.show()

## 7. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_DIR = Path("dataset/split")
SPLITS = ["train", "val", "test"]
for split in SPLITS:
    for cls in CLASS_NAMES:
        (SPLIT_DIR / split / cls).mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png"}

for class_name in CLASS_NAMES:
    class_dir = RAW_DIR / class_name
    all_imgs = [p for p in class_dir.iterdir() if p.suffix.lower() in VALID_EXTS]

    if len(all_imgs) < 5:
        print(f"⚠️  {class_name}: only {len(all_imgs)} images — skipping split")
        continue

    train_imgs, temp_imgs = train_test_split(all_imgs, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.5, random_state=42)

    for imgs, split in [(train_imgs, "train"), (val_imgs, "val"), (test_imgs, "test")]:
        for img_path in imgs:
            dest = SPLIT_DIR / split / class_name / img_path.name
            shutil.copy2(img_path, dest)

    print(f"{class_name:<25} train={len(train_imgs):>4}  val={len(val_imgs):>3}  test={len(test_imgs):>3}")

print("\n✅ Dataset split complete!")

## 8. Config & Data Generators

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
IMG_SIZE   = 224          # EfficientNetB0 native input size
BATCH_SIZE = 32
EPOCHS_FROZEN  = 15       # Phase 1: train only top layers
EPOCHS_FINETUNE = 20      # Phase 2: fine-tune top conv blocks
LR_FROZEN  = 1e-3
LR_FINETUNE = 1e-5
SEED = 42

# ── Augmentation (train only) ────────────────────────────────────────────────
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

val_test_datagen = ImageDataGenerator(rescale=1.0 / 255)

# ── Generators ───────────────────────────────────────────────────────────────
train_gen = train_datagen.flow_from_directory(
    SPLIT_DIR / "train",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=SEED,
)

val_gen = val_test_datagen.flow_from_directory(
    SPLIT_DIR / "val",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

test_gen = val_test_datagen.flow_from_directory(
    SPLIT_DIR / "test",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False,
)

class_indices = train_gen.class_indices
print("Class indices:", class_indices)

# Save label map
label_map = {v: k for k, v in class_indices.items()}
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)
print("Label map saved to label_map.json")

## 9. Model Architecture (EfficientNetB0 + Custom Head)

In [ ]:
def build_model(num_classes: int, img_size: int = 224, frozen: bool = True):
    """
    EfficientNetB0 backbone + custom classification head.
    frozen=True  → only top layers trainable (Phase 1)
    frozen=False → fine-tune top conv blocks (Phase 2)
    """
    base = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(img_size, img_size, 3),
    )
    base.trainable = not frozen

    if not frozen:
        # Unfreeze only top 30 layers for fine-tuning
        for layer in base.layers[:-30]:
            layer.trainable = False

    inputs = keras.Input(shape=(img_size, img_size, 3))
    x = base(inputs, training=not frozen)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    return keras.Model(inputs, outputs)


model = build_model(NUM_CLASSES, IMG_SIZE, frozen=True)
model.summary()

trainable_params = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f"\nTrainable params: {trainable_params:,}")

## 10. Phase 1 — Train Top Layers (Frozen Backbone)

In [ ]:
os.makedirs("checkpoints", exist_ok=True)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FROZEN),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_phase1 = [
    ModelCheckpoint(
        "checkpoints/best_phase1.keras",
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_loss", patience=5, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    CSVLogger("training_phase1_log.csv"),
]

print("Phase 1: Training with frozen backbone...")
history1 = model.fit(
    train_gen,
    epochs=EPOCHS_FROZEN,
    validation_data=val_gen,
    callbacks=callbacks_phase1,
    verbose=1,
)
print("\n✅ Phase 1 complete!")

## 11. Phase 2 — Fine-Tune Top Conv Blocks

In [ ]:
# Reload best weights from Phase 1
model.load_weights("checkpoints/best_phase1.keras")

# Unfreeze top 30 layers of backbone
base_model = model.layers[1]  # EfficientNetB0
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_params = sum([np.prod(v.shape) for v in model.trainable_variables])
print(f"Fine-tune trainable params: {trainable_params:,}")

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_FINETUNE),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks_phase2 = [
    ModelCheckpoint(
        "checkpoints/best_final.keras",
        monitor="val_accuracy", save_best_only=True, verbose=1
    ),
    EarlyStopping(
        monitor="val_loss", patience=7, restore_best_weights=True, verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss", factor=0.3, patience=4, min_lr=1e-8, verbose=1
    ),
    CSVLogger("training_phase2_log.csv"),
]

print("Phase 2: Fine-tuning top conv blocks...")
history2 = model.fit(
    train_gen,
    epochs=EPOCHS_FINETUNE,
    validation_data=val_gen,
    callbacks=callbacks_phase2,
    verbose=1,
)
print("\n✅ Phase 2 complete!")

## 12. Accuracy & Loss Plots

In [ ]:
def merge_histories(h1, h2):
    """Concatenate Phase 1 + Phase 2 history dicts."""
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history[key]
    return merged

merged = merge_histories(history1, history2)
phase1_len = len(history1.history["accuracy"])
total_epochs = len(merged["accuracy"])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("Training Curves — Food Plate/Bowl Detector", fontsize=15, fontweight="bold")

epochs_range = range(1, total_epochs + 1)
colors = {"train": "#2196F3", "val": "#FF5722", "vline": "#4CAF50"}

# ── Accuracy ────────────────────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_range, merged["accuracy"],     label="Train Acc",  color=colors["train"], linewidth=2)
ax.plot(epochs_range, merged["val_accuracy"], label="Val Acc",    color=colors["val"],   linewidth=2, linestyle="--")
ax.axvline(phase1_len + 0.5, color=colors["vline"], linestyle=":", linewidth=1.5, label=f"Fine-tune start (ep {phase1_len+1})")
ax.fill_between(epochs_range, merged["accuracy"], merged["val_accuracy"],
                alpha=0.08, color=colors["val"])
ax.set_title("Accuracy", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Accuracy")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)

# Best val accuracy annotation
best_val_acc_ep = np.argmax(merged["val_accuracy"]) + 1
best_val_acc    = max(merged["val_accuracy"])
ax.annotate(f"Best: {best_val_acc:.3f}",
            xy=(best_val_acc_ep, best_val_acc),
            xytext=(best_val_acc_ep + 1, best_val_acc - 0.05),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=9)

# ── Loss ────────────────────────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_range, merged["loss"],     label="Train Loss", color=colors["train"], linewidth=2)
ax.plot(epochs_range, merged["val_loss"], label="Val Loss",   color=colors["val"],   linewidth=2, linestyle="--")
ax.axvline(phase1_len + 0.5, color=colors["vline"], linestyle=":", linewidth=1.5, label=f"Fine-tune start (ep {phase1_len+1})")
ax.set_title("Loss", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Categorical Cross-Entropy")
ax.legend()
ax.grid(True, alpha=0.3)

# Best val loss annotation
best_val_loss_ep = np.argmin(merged["val_loss"]) + 1
best_val_loss    = min(merged["val_loss"])
ax.annotate(f"Best: {best_val_loss:.3f}",
            xy=(best_val_loss_ep, best_val_loss),
            xytext=(best_val_loss_ep + 1, best_val_loss + 0.05),
            arrowprops=dict(arrowstyle="->", color="black"),
            fontsize=9)

plt.tight_layout()
plt.savefig("accuracy_loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved as accuracy_loss_curves.png")

## 13. Test Set Evaluation

In [ ]:
# Load best weights before evaluation
model.load_weights("checkpoints/best_final.keras")

test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f"\n{'='*40}")
print(f"  Test Loss    : {test_loss:.4f}")
print(f"  Test Accuracy: {test_acc:.4f}  ({test_acc*100:.2f}%)")
print(f"{'='*40}")

## 14. Classification Report & Confusion Matrix

In [ ]:
# Predict
test_gen.reset()
y_pred_proba = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = test_gen.classes

idx_to_class = {v: k for k, v in test_gen.class_indices.items()}
class_labels = [idx_to_class[i] for i in range(NUM_CLASSES)]

# Classification Report
print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=class_labels))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Confusion Matrix", fontsize=14, fontweight="bold")

for ax, data, fmt, title in [
    (axes[0], cm,      "d",    "Raw Counts"),
    (axes[1], cm_norm, ".2f",  "Normalized"),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap="Blues",
        xticklabels=class_labels, yticklabels=class_labels,
        ax=ax, linewidths=0.5
    )
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    plt.setp(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
    plt.setp(ax.get_yticklabels(), rotation=0, fontsize=9)

plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

## 15. Per-Class Confidence Analysis

In [ ]:
# Average confidence per true class
fig, ax = plt.subplots(figsize=(10, 4))
conf_by_class = []
for i, cls in enumerate(class_labels):
    mask = y_true == i
    if mask.sum() > 0:
        avg_conf = y_pred_proba[mask, i].mean()
        conf_by_class.append(avg_conf)
    else:
        conf_by_class.append(0)

bars = ax.bar(class_labels, conf_by_class, color=sns.color_palette("viridis", NUM_CLASSES))
ax.set_ylim(0, 1.05)
ax.set_title("Average Predicted Confidence per Class", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean Confidence")
plt.xticks(rotation=30, ha="right")
for bar, conf in zip(bars, conf_by_class):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{conf:.2f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig("confidence_per_class.png", dpi=150)
plt.show()

## 16. Save Model & History

In [ ]:
import zipfile

# ── Save full model ──────────────────────────────────────────────────────────
model.save("food_detector_final.keras")
print("✅ Full model saved: food_detector_final.keras")

# ── Save as TF SavedModel (for TFLite / TF Serving) ─────────────────────────
model.export("food_detector_savedmodel")
print("✅ SavedModel saved: food_detector_savedmodel/")

# ── Save training history ────────────────────────────────────────────────────
full_history = {}
for key in merged:
    full_history[key] = [float(v) for v in merged[key]]
full_history["phase1_epochs"] = phase1_len
full_history["total_epochs"]  = total_epochs
full_history["best_val_accuracy"] = float(best_val_acc)
full_history["best_val_loss"]     = float(best_val_loss)
full_history["test_accuracy"]     = float(test_acc)
full_history["test_loss"]         = float(test_loss)
full_history["class_names"]       = CLASS_NAMES

with open("training_history.json", "w") as f:
    json.dump(full_history, f, indent=2)
print("✅ Training history saved: training_history.json")

# ── Save label map ───────────────────────────────────────────────────────────
with open("label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)
print("✅ Label map saved: label_map.json")

# ── Bundle outputs into zip ──────────────────────────────────────────────────
output_files = [
    "food_detector_final.keras",
    "training_history.json",
    "label_map.json",
    "accuracy_loss_curves.png",
    "confusion_matrix.png",
    "confidence_per_class.png",
    "class_distribution.png",
    "training_phase1_log.csv",
    "training_phase2_log.csv",
]

with zipfile.ZipFile("food_detector_outputs.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in output_files:
        if os.path.exists(fp):
            zf.write(fp)

print("\n✅ All outputs bundled: food_detector_outputs.zip")
print("\n── Summary ──────────────────────────────────────────")
print(f"  Best Val Accuracy : {best_val_acc*100:.2f}%")
print(f"  Best Val Loss     : {best_val_loss:.4f}")
print(f"  Test Accuracy     : {test_acc*100:.2f}%")
print(f"  Test Loss         : {test_loss:.4f}")

## 17. Load & Infer (Inference Utility)

In [ ]:
from tensorflow.keras.preprocessing import image as keras_image

def load_model_and_predict(model_path: str, label_map_path: str, img_path: str):
    """Load a saved model and predict on a single image."""
    loaded_model = keras.models.load_model(model_path)
    with open(label_map_path) as f:
        lmap = json.load(f)

    img = keras_image.load_img(img_path, target_size=(IMG_SIZE, IMG_SIZE))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = loaded_model.predict(arr, verbose=0)[0]
    top_idx = int(np.argmax(probs))
    label   = lmap[str(top_idx)]
    conf    = float(probs[top_idx])

    print(f"Predicted class : {label}")
    print(f"Confidence      : {conf*100:.1f}%")
    print("\nAll probabilities:")
    for idx, prob in sorted(enumerate(probs), key=lambda x: -x[1]):
        print(f"  {lmap[str(idx)]:<25} {prob*100:.1f}%")
    return label, conf


# ── Demo: pick a random test image and classify it ───────────────────────────
all_test_imgs = list(Path("dataset/split/test").rglob("*.jpg")) + \
                list(Path("dataset/split/test").rglob("*.png"))

if all_test_imgs:
    sample_img = random.choice(all_test_imgs)
    true_label = sample_img.parent.name
    print(f"\nTest image  : {sample_img.name}")
    print(f"True label  : {true_label}")
    print("-" * 40)
    load_model_and_predict("food_detector_final.keras", "label_map.json", str(sample_img))

    # Show the image
    img = mpimg.imread(str(sample_img))
    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f"True: {true_label}", fontsize=11)
    plt.axis("off")
    plt.tight_layout()
    plt.show()

## 18. Final Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════╗
║        🍽️  Food Plate/Bowl Detector — Done!           ║
╠══════════════════════════════════════════════════════╣
║  Model       : EfficientNetB0 + Custom Head          ║
║  Classes     : 6 (plate, bowl, neither, empties,     ║
║                   hard negatives)                    ║
║  Training    : Phase 1 (frozen) + Phase 2 (finetune) ║
╠══════════════════════════════════════════════════════╣
║  Saved Files:                                        ║
║  • food_detector_final.keras   ← full model          ║
║  • food_detector_savedmodel/   ← TF SavedModel       ║
║  • training_history.json       ← metrics + meta      ║
║  • label_map.json              ← index → class name  ║
║  • accuracy_loss_curves.png    ← training plots      ║
║  • confusion_matrix.png        ← evaluation          ║
║  • food_detector_outputs.zip   ← all bundled         ║
╚══════════════════════════════════════════════════════╝
""")

with open("training_history.json") as f:
    h = json.load(f)

print(f"  Best Val Accuracy : {h['best_val_accuracy']*100:.2f}%")
print(f"  Best Val Loss     : {h['best_val_loss']:.4f}")
print(f"  Test Accuracy     : {h['test_accuracy']*100:.2f}%")
print(f"  Test Loss         : {h['test_loss']:.4f}")